# Simulate P6 validation data for the Essen Bayes short course

**Purpose.** Create a *synthetic* participant-level dataset for a teaching demonstration inspired by the P6 validation analysis in Thielscher et al. (2026), *Harmonizing the stimulation dose of focal transcranial direct current stimulation across target sites*.

The paper reports that:

- P6 is the **right cerebellar target associated with eyeblink conditioning**.
- The intended group-average electric-field magnitude is **0.2 V/m**.
- The montage was validated in an independent **Sample 2 with 53 participants**.
- P6 values were distributed around the 0.2 V/m target and had comparatively high inter-individual variability.

The paper does **not** publish the 53 participant-level P6 values. The data generated here are therefore **simulated teaching data, not reconstructed or imputed original data**.

This notebook is intentionally separate from the analysis notebook. If real participant-level data become available later, the analysis notebook can use them without changing the analysis code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEED = 20260920
N = 53
TARGET = 0.20

# Teaching choices chosen to resemble the qualitative appearance of P6
# in Figure 2E. These are NOT values reported by the paper.
POPULATION_MEAN = 0.205
POPULATION_CV = 0.28


## Simulation model

Electric-field magnitude must be positive and the P6 violin in the paper shows substantial person-to-person spread. For the teaching dataset we therefore use a **log-normal distribution**.

The exact distributional choice is not important for the participant-facing analysis. Its only job is to produce a plausible synthetic dataset with:

1. 53 positive values,
2. a sample center near 0.2 V/m,
3. enough between-person variability to make the distinction between the **group mean** and the **field expected for a new person** visible.

In [ ]:
rng = np.random.default_rng(SEED)

sigma_log = np.sqrt(np.log(1 + POPULATION_CV**2))
mu_log = np.log(POPULATION_MEAN) - 0.5 * sigma_log**2

efield = rng.lognormal(mean=mu_log, sigma=sigma_log, size=N)

data = pd.DataFrame({
    "participant_id": [f"S2_{i:02d}" for i in range(1, N + 1)],
    "project": "P6",
    "target_region": "right cerebellum (Crus I / lobule VI)",
    "task_context": "eyeblink conditioning",
    "efield_v_per_m": efield,
    "synthetic": True,
})

data.head()


## Check the resulting synthetic dataset

In [ ]:
data["efield_v_per_m"].describe(percentiles=[.25, .5, .75])


In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(np.arange(1, N + 1), data["efield_v_per_m"], alpha=0.75)
plt.axhline(TARGET, linestyle="--", label="Target = 0.20 V/m")
plt.xlabel("Synthetic participant")
plt.ylabel("Electric-field magnitude (V/m)")
plt.title("Synthetic P6 validation data")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(data["efield_v_per_m"], bins=12)
plt.axvline(TARGET, linestyle="--", label="Target = 0.20 V/m")
plt.xlabel("Electric-field magnitude (V/m)")
plt.ylabel("Count")
plt.title("Distribution of synthetic P6 values")
plt.legend()
plt.show()


## Export

The analysis notebook expects a CSV with a column named `efield_v_per_m`.  
To replace these synthetic data with real data later, create a CSV with the same column name.

In [ ]:
output_path = Path("synthetic_p6_validation.csv")
data.to_csv(output_path, index=False)
print(f"Saved {output_path.resolve()}")
print(f"n = {len(data)}")
print(f"sample mean = {data['efield_v_per_m'].mean():.3f} V/m")
print(f"sample SD   = {data['efield_v_per_m'].std(ddof=1):.3f} V/m")
